# Milestone 1 — arXiv data collection

**Snapshot:** 25 October 2026 (UTC), generated from Bao’s local verified crawl cache. **Scope:** assigned range `2508.02872`–`2508.07871` (5,000 IDs). This is an evidence report for one member, not a claim that the five-member project is complete.

The notebook reads only `data/bao_snapshot.json`, a small aggregate snapshot. It contains no paper titles, authors, reference payloads, source files, or credentials. Rebuilds do not crawl the Internet.

The course requirements specify arXiv API discovery, source retrieval from `export.arxiv.org/src/{id}vN`, Semantic Scholar references, and generated report figures. See the [course crawling specification](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).


In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
from IPython.display import display, Markdown, HTML

DATA = Path("data/bao_snapshot.json")
if not DATA.exists():
    DATA = Path("../data/bao_snapshot.json")
report = json.loads(DATA.read_text(encoding="utf-8"))
assert report["format"] == 1
counts = report["counts"]
print(f"Snapshot: {report['snapshot_at']} | source commit: {report['source_commit'][:12]}")


## Collection method and reproducibility

The local range runner discovered metadata for the assigned IDs, enumerated the available versions, and fetched source archives through the integrated crawler (`engine=lieu`). The cache records per-task status and checksums; the report generator revalidated the manifest and receipt-backed source files before writing this aggregate. Semantic Scholar reference retrieval was paginated and stored locally. Exported Parquet tables were checked against the current two-table crawling schema.

The final corpus accumulated over multiple runs and code revisions, so its total elapsed time cannot be interpreted as one fresh-run benchmark. The separate 59-ID retry used the upstream standalone trial and did not recover additional publications.


In [ ]:
summary = pd.DataFrame([
    ("Assigned IDs", counts["assigned"], counts["assigned"], counts["assigned"]),
    ("Discovered publications", counts["discovered"], counts["assigned"], counts["discovered"]/counts["assigned"]),
    ("Complete publications", counts["complete"], counts["assigned"], counts["complete"]/counts["assigned"]),
    ("Expected source versions", counts["expected_versions"], None, None),
    ("TeX source versions", counts["success"], counts["expected_versions"], counts["success"]/counts["expected_versions"]),
    ("Non-TeX versions", counts["non_tex"], counts["expected_versions"], counts["non_tex"]/counts["expected_versions"]),
    ("Unavailable source versions", counts["unavailable"], counts["expected_versions"], counts["unavailable"]/counts["expected_versions"]),
    ("Reference retrieval success", counts["references_success"], counts["assigned"], counts["references_success"]/counts["assigned"]),
], columns=["Measure", "Numerator", "Denominator", "Rate"])
summary["Rate"] = summary["Rate"].map(lambda x: "—" if pd.isna(x) else f"{x:.2%}")
display(summary)


In [ ]:
complete_rate=counts["complete"]/counts["assigned"]
tex_rate=counts["success"]/counts["expected_versions"]
nontex_rate=counts["non_tex"]/counts["expected_versions"]
unavailable_rate=counts["unavailable"]/counts["expected_versions"]
ref_rate=counts["references_success"]/counts["assigned"]
display(Markdown(
    f"{counts['discovered']:,} / {counts['assigned']:,} IDs were discovered. "
    f"{counts['complete']:,} / {counts['assigned']:,} publications "
    f"({complete_rate:.2%}) were complete in the local cache. Across "
    f"{counts['expected_versions']:,} expected versions, {counts['success']:,} "
    f"({tex_rate:.2%}) had verified TeX source, {counts['non_tex']:,} "
    f"({nontex_rate:.2%}) were valid non-TeX submissions, and "
    f"{counts['unavailable']:,} ({unavailable_rate:.2%}) were unavailable. "
    f"Reference retrieval succeeded for {counts['references_success']:,} / "
    f"{counts['assigned']:,} publications ({ref_rate:.2%}). A completed reference "
    "request can validly return zero references; it is distinct from a failed request."
))


The following interpretation is generated from the same aggregate values as the table above.


## Versions and reference yield

The distribution below is generated from the snapshot. Reference counts describe successful Semantic Scholar responses, including responses with no references.


In [ ]:
versions = pd.DataFrame(report["versions_distribution"])
fig = px.bar(versions, x="versions", y="publications",
             labels={"versions":"Versions reported by arXiv", "publications":"Publications"},
             title="Publications by number of arXiv versions")
fig.update_layout(template="plotly_white")
display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))

ref = report["references"]
print(f"Reference records retained locally: {ref['all_records']:,}")
print(f"Records with an arXiv identifier before pair deduplication: {ref['with_arxiv_before_dedup']:,}")
print(f"Unique (citing paper, cited arXiv ID) pairs: {ref['unique_arxiv_pairs']:,}")
print(f"Successfully empty reference responses: {ref['successfully_empty_publications']:,}")
print(f"Mean references per successful publication: {ref['per_successful_publication']['mean']:.2f}; median: {ref['per_successful_publication']['median']:.0f}")
hist = pd.DataFrame(ref["histogram"])
hist["bin"] = hist.apply(lambda r: f"{r['lower']}–{int(r['upper_exclusive'])-1}" if pd.notna(r['upper_exclusive']) else f"{r['lower']}+", axis=1)
fig = px.bar(hist, x="bin", y="count", labels={"bin":"References per successful publication", "count":"Publications"}, title="Reference count distribution")
fig.update_layout(template="plotly_white")
display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False)))


In [ ]:
display(Markdown(
    f"The local normalized `references` Parquet contains **{report['checks']['tables']['references']['rows']:,} rows**; "
    f"the raw successful responses contain **{ref['all_records']:,} reference records**. "
    f"The raw count includes references without an arXiv ID. The Parquet includes only records "
    f"with an arXiv ID and publications represented in the export. The **{ref['unique_arxiv_pairs']:,}** "
    "unique arXiv pairs also include successful publications whose source was incomplete, "
    "so the figures use different scopes."
))


The raw successful responses include records without an arXiv ID. The exported Parquet includes only records with an arXiv ID and is restricted to publications represented in the export. These figures intentionally use different definitions. Unique arXiv pairs also include successful publications whose source was incomplete, so they need not equal the exported table row count.


## Storage and output tables

“Retained source bytes” is the logical sum of extracted `.tex` and `.bib` files verified in the cache. It is **not** the original compressed archive size. The original pre-extraction byte totals were not recorded for this integrated corpus, so a reduction ratio and average before/after comparison cannot be computed honestly. “Active output” is a snapshot of current logical file sizes, not peak disk usage.


In [ ]:
storage = report["storage"]
size_table = pd.DataFrame([
    ("Raw source, metadata and references", storage["active_output_bytes"]["raw"] + storage["active_output_bytes"]["metadata"] + storage["active_output_bytes"]["references"]),
    ("Tables", storage["active_output_bytes"]["tables"]),
    ("Retained TeX and Bib files", storage["retained_tex_bib_bytes"]),
], columns=["Output", "Bytes"])
display(size_table.style.format({"Bytes":"{:,.0f}"}))
print("Mean retained TeX/Bib bytes per complete publication:", f"{storage['complete_publication_retained_bytes']['mean']:,.0f}")
print("Before-extraction mean: NOT RECORDED; reduction: NOT COMPUTABLE")

tables = report["checks"]["tables"]
rows=[]
for name, t in tables.items():
    rows.append({"Table":name, "Valid":t.get("valid"), "Rows":t.get("rows"), "Duplicate keys":t.get("duplicate_keys"), "IDs outside range":t.get("outside_range"), "Columns":", ".join(f["name"] for f in t.get("fields",[]))})
display(pd.DataFrame(rows))


## Performance and benchmark status

The course asks for a CPU-only Google Colab benchmark, including end-to-end wall time, mean time per publication, discovery time, sampled mean and peak RAM across the full process tree, peak disk usage, and final output size. These figures were **not measured** for this corpus. The report leaves them missing rather than estimating them from a two-paper local smoke run.

A prior 59-ID retry on a local macOS machine took 723.7 seconds (59 tasks, zero publications recovered); it is an operational retry, not a fresh-crawl benchmark and not a Colab result. Two earlier smoke runs are likewise excluded from corpus-wide averages.


In [ ]:
benchmark = report["benchmark"]
perf = pd.DataFrame([
    ("Total end-to-end wall time", benchmark["total_wall_seconds"]),
    ("Mean seconds per publication", benchmark["mean_seconds_per_publication"]),
    ("Metadata discovery seconds", benchmark["discovery_seconds"]),
    ("Peak RAM (process tree)", benchmark["peak_rss_bytes"]),
    ("Mean RAM (sampled process tree)", benchmark["mean_rss_bytes"]),
    ("Peak disk usage", benchmark["peak_disk_bytes"]),
], columns=["Requested metric", "Measured value"])
perf["Measured value"] = perf["Measured value"].map(lambda x: "Not measured" if x is None else x)
display(perf)
print("Testbed:", benchmark["testbed"])


## Data quality, failures, and limitations

The reporting snapshot validated 17,751 input files by recorded checksums and confirmed the manifest did not change during report generation. The `papers` Parquet has 4,941 rows, no duplicate IDs, and no IDs outside Bao's range. The `references` Parquet has 93,284 rows, no duplicate composite keys, and no citing IDs outside the range. No unexpected raw source file extensions were found by the report hygiene check.

The 48 unavailable source tasks and 15 failed reference tasks are recorded separately in the snapshot with ID, task kind, status, and HTTP status. A 404 is reported as unavailable/error, never as a valid empty submission or empty reference list. Individual failures are available in `data/bao_snapshot.json`; this public report intentionally avoids publishing source text or reference payloads.

Limitations: this is one member's range; cross-member overlap/schema consistency and the final merged dataset were not audited. The corpus is incomplete. The report does not establish byte-for-byte equality against source archives no longer retained, and does not establish peak resource usage. Parser/Milestone 2 results are not included yet.


The aggregate hash covers relative input paths and SHA-256 digests. No credentials or raw bibliographic content is included. See `checklist.md` for item-by-item evidence and outstanding checks.


In [ ]:
display(Markdown(
    f"- Member/range: {report['member']}, `{report['range']['first']}`–`{report['range']['last']}`\n"
    f"- Snapshot time (UTC): `{report['snapshot_at']}`\n"
    f"- Source repository commit: `{report['source_commit']}`\n"
    f"- Last worker commit: `{report['crawl']['last_worker_commit']}`\n"
    f"- Verified inputs: {report['provenance']['verified_input_files']:,}; aggregate hash: `{report['provenance']['aggregate_input_hash']}`\n"
    f"- Manifest SHA-256: `{report['provenance']['manifest_sha256']}`"
))
